## Refold Complex

### Refold with target templates

In [6]:
# import functions
import numpy as np
import jax
import re
import os
import json
import pyrosetta as pr
import pandas as pd
from Bio import PDB
from Bio.PDB import PDBIO, Select
from Bio.SeqUtils import seq1
from colabdesign import mk_af_model, clear_mem
from colabdesign.shared.utils import copy_dict
from functions.pyrosetta_utils import pr_relax, unaligned_rmsd, align_pdbs
from functions.biopython_utils import target_pdb_rmsd

from myScripts.complexRefold import predict_complex, predict_binder, SelectChain, separate_complex
pr.init('-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball /hpf/projects/mtyers/ningrui/NXBindCraft/functions/DAlphaBall.gcc -corrections::beta_nov16 true -relax:default_repeats 1')


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2025.06+release.029c6a159b896477003a14f78f472d4cd2cead46 2025-02-04T15:14:13] retrieved from: http://www.pyrosetta.org


In [7]:
main_folder = '/hpf/projects/mtyers/ningrui/NXBindCraft'
trial_name = 'testRefold'
params_path = '/hpf/projects/mtyers/ningrui//BindCraft'


In [8]:
num_recycles_validation = 3
multimer_validation = False
predicted_initial_guess = False
predict_bigbang = False
rm_template_seq_predict = False
rm_template_sc_predict = False
complex_prediction_models = [0,1]
binder_prediction_models = [0,1,2,3,4]

filter_name = 'default_filters.json'
filter_path = os.path.join('/hpf/projects/mtyers/ningrui/NXBindCraft/settings_filters', filter_name)
with open(filter_path, 'r') as file:
    filters = json.load(file)


In [9]:
complex_list = ['5zng', '6a6i', '6gs2', '6h4b', '6if2']
rmsds = ['HotspotRMSD', 'Target_RMSD', 'Binder_RMSD', 'Pred_Binder_RMSD', 'Pass_base_af2_filters']
rmsd_df = pd.DataFrame(index=complex_list, columns=rmsds)

In [ ]:
for complex_name in complex_list:
    target_pdb, binder_seq, target_chain, binder_chain = separate_complex(complex_name)
    gt_complex_pdb = os.path.join(main_folder, 'myTrials/testRefold/Targets', complex_name+'.pdb')
    print(f'Complex: {complex_name}')

    # compile complex prediction model
    complex_prediction_model = mk_af_model(protocol = 'binder',
                                        num_recycles = num_recycles_validation,
                                        data_dir = params_path,
                                        use_multimer = multimer_validation,
                                        use_initial_guess = predicted_initial_guess,
                                        use_initial_atom_pos = predict_bigbang)

    complex_prediction_model.prep_inputs(pdb_filename=target_pdb, 
                                     chain='A', 
                                     binder_len=len(binder_seq),
                                     rm_target_seq=rm_template_seq_predict, 
                                     rm_target_sc=rm_template_sc_predict)
    print('Finish prepare complex model')
    
    # compile binder monomer prediction model
    binder_prediction_model = mk_af_model(protocol='hallucination',
                                      use_templates = False,
                                      initial_guess=False,
                                      use_initial_atom_pos=False,
                                      num_recycles=num_recycles_validation,
                                      data_dir=params_path,
                                      use_multimer=multimer_validation)
    binder_prediction_model.prep_inputs(length=len(binder_seq))
    print('Finish prepare binder model')

    # complex stats
    complex_statistics, pass_af2_filters = predict_complex(prediction_model=complex_prediction_model,
                                                       binder_sequence=binder_seq, 
                                                       complex_name=complex_name,
                                                       prediction_models=complex_prediction_models,
                                                       num_recycles_validation=num_recycles_validation,
                                                       filters=filters)

    print(complex_statistics)
    # if not pass af2 filters, noted in dataframe; but also do basic scoring
    # scoring complex rmsds
    # select the best model with highest plddt
    pass_af2_keys = [k for k, v in complex_statistics.items() if v['pass_af2_filters']]
    if pass_af2_keys:
        best_complex_pred_model = max(pass_af2_keys, key=lambda k: complex_statistics[k]['pLDDT'])
    else:
        best_complex_pred_model = max(complex_statistics, key=lambda k: complex_statistics[k]['pLDDT'])

    best_complex_pdb = os.path.join(main_folder, 'myTrials/testRefold', 'refoldPDB/refold', f'{complex_name}_model{best_complex_pred_model}.pdb')
    if not os.path.exists(best_complex_pdb):
        print('Predicted complex structure not exists')
    else:
        rmsd_site = unaligned_rmsd(gt_complex_pdb, best_complex_pred_model, binder_chain, 'B')
        target_rmsd = target_pdb_rmsd(best_complex_pdb, target_pdb, target_chain)

    rmsd_df.loc[complex_name, 'HotspotRMSD'] = rmsd_site
    rmsd_df.loc[complex_name, 'Target_RMSD'] = target_rmsd
    rmsd_df.loc[complex_name, 'Pass_base_af2_filters'] = complex_statistics[best_complex_pred_model]['pass_af2_filters']

    # scoring binder rmsds
    # NOTE: paper only used template based model [0,1], here used [0-4]
    binder_statistics = predict_binder(prediction_model=binder_prediction_model,
                                   binder_sequence=binder_seq,
                                   complex_name=complex_name,
                                   gt_pdb=gt_complex_pdb,
                                   binder_chain=binder_chain,
                                   num_recycles_validation=num_recycles_validation,
                                   prediction_models=binder_prediction_models)
    
    best_binder_pred_model = max(binder_statistics, key=lambda k: binder_statistics[k]['pLDDT'])
    best_binder_pdb = os.path.join(main_folder, "myTrials/testRefold", 'refoldPDB/refold_binder', f'{complex_name}_binder_model{best_binder_pred_model}.pdb')
    if not os.path.exists(best_binder_pdb):
        print('Predicted binder alone structure not exists')
    else:
        rmsd_binder = unaligned_rmsd(gt_complex_pdb, best_binder_pdb, binder_chain, 'A')
    
    rmsd_df.loc[complex_name, 'Binder_RMSD'] = rmsd_binder

    # get rmsd between predicted binder alone vs. predicted complex binder (NOTE: structure only)
    # 1: align complex binder with predicted binder
    align_pdbs(best_complex_pdb, best_binder_pdb, 'B', 'A')
    rmsd_pre_binder = unaligned_rmsd(best_complex_pdb, best_binder_pdb, 'B', 'A')
    # 2: align the binder alone back with the ground truth complex structure
    align_pdbs(gt_complex_pdb, best_binder_pdb, binder_chain, 'A')

    rmsd_df.loc[complex_name, 'Pred_Binder_RMSD'] = rmsd_pre_binder

